<a href="https://colab.research.google.com/github/lricci03/challengedata/blob/main/CFM2024-new/cfm2024-feature-engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature engeneering

Create new features that encode the 100 observations per obs_id, such as mean price, min price, max price, number of total trades, etc.

In [ ]:
from google.colab import drive

# drive.flush_and_unmount()
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/Challengedata/Xtrain.parquet")
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080000 entries, 0 to 16079999
Data columns (total 12 columns):
 #   Column    Dtype  
---  ------    -----  
 0   obs_id    int64  
 1   venue     int64  
 2   order_id  int64  
 3   action    object 
 4   side      object 
 5   price     float64
 6   bid       float64
 7   ask       float64
 8   bid_size  int64  
 9   ask_size  int64  
 10  trade     bool   
 11  flux      int64  
dtypes: bool(1), float64(3), int64(6), object(2)
memory usage: 2.6 GB


In [ ]:
df.shape
df.memory_usage(deep=True).sum() / 1024**3

np.float64(2.5907905139029026)

Reduces dtypes. Not price columns as they could contain small prices/price differences.

In [ ]:
df["venue"] = df["venue"].astype("int8")
df["action"] = df["action"].astype("category")
df["side"] = df["side"].astype("category")

df["bid_size"] = pd.to_numeric(df["bid_size"], downcast="integer")
df["ask_size"] = pd.to_numeric(df["ask_size"], downcast="integer")
df["flux"] = pd.to_numeric(df["flux"], downcast="integer")
df["order_id"] = pd.to_numeric(df["order_id"], downcast="integer")

In [ ]:
df.shape
df.memory_usage(deep=True).sum() / 1024**3

np.float64(0.7338082399219275)

Sanity checks

In [ ]:
df["obs_id"].nunique()

160800

In [ ]:
df.groupby("obs_id").size().value_counts()

,count
100,160800


We implement the following features:

* Price/spread structure:

  * median_price
  * median_mid_price

  * mean_spread
  * median_spread
  * std_spread
  * min_spread
  * max_spread

* Book size:
  * mean_bid_size
  * median_bid_size
  * mean_ask_size
  * median_ask_size

* Imbalance:
  * mean_obi
  * median_obi
  * std_obi

* Event composition:
  * trade_frequency
  * cancel_ratio
  * unique_orders

* Flux:
  * mean_abs_flux: typical event size
  * median_abs_flux
  * mean_flux: net-liquid adding or removing
  * std_flux

* Venues:
  * venue_x_ratio

* Actions:
  * add_ratio
  * delete_ratio
  * update_ratio




Create tick-level helper columns

In [ ]:
import numpy as np
import pandas as pd


df["mid_price"] = (df["bid"] + df["ask"]) / 2
df["spread"] = df["ask"] - df["bid"]

df["obi"] = (
    (df["bid_size"] - df["ask_size"])
    / (df["bid_size"] + df["ask_size"])
)

df["abs_flux"] = df["flux"].abs()

df["is_trade"] = df["trade"].astype("int8")
df["is_cancel"] = (
    (df["action"] == "D") & (~df["trade"])
).astype("int8")

df["is_add"] = (df["action"] == "A").astype("int8")
df["is_delete"] = (df["action"] == "D").astype("int8")
df["is_update"] = (df["action"] == "U").astype("int8")

Aggregate observations by obs_id

In [ ]:
g = df.groupby("obs_id", sort=False)

Aggregate: this gives one row per obs_id

In [ ]:
features = g.agg(
    # Price / spread structure
    median_price=("price", "median"),
    median_mid_price=("mid_price", "median"),

    mean_spread=("spread", "mean"),
    median_spread=("spread", "median"),
    std_spread=("spread", "std"),
    min_spread=("spread", "min"),
    max_spread=("spread", "max"),

    # Book size
    mean_bid_size=("bid_size", "mean"),
    median_bid_size=("bid_size", "median"),
    mean_ask_size=("ask_size", "mean"),
    median_ask_size=("ask_size", "median"),

    # Imbalance
    mean_obi=("obi", "mean"),
    median_obi=("obi", "median"),
    std_obi=("obi", "std"),

    # Event composition
    trade_frequency=("is_trade", "mean"),
    cancel_ratio=("is_cancel", "mean"),
    unique_orders=("order_id", "nunique"),

    # Flux
    mean_abs_flux=("abs_flux", "mean"),
    median_abs_flux=("abs_flux", "median"),
    mean_flux=("flux", "mean"),
    std_flux=("flux", "std"),

    # Actions
    add_ratio=("is_add", "mean"),
    delete_ratio=("is_delete", "mean"),
    update_ratio=("is_update", "mean"),
)

Venue ratios. They produce one column per venue

In [ ]:
venue_ratios = (
    g["venue"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)

venue_ratios.columns = [
    f"venue_{v}_ratio"
    for v in venue_ratios.columns
]

In [ ]:
features = pd.concat(
    [features, venue_ratios],
    axis=1
).reset_index()

In [ ]:
features.shape
features.head()

,obs_id,median_price,median_mid_price,mean_spread,median_spread,std_spread,min_spread,max_spread,mean_bid_size,median_bid_size,...,std_flux,add_ratio,delete_ratio,update_ratio,venue_4_ratio,venue_1_ratio,venue_5_ratio,venue_2_ratio,venue_0_ratio,venue_3_ratio
0,0,0.00,0.010,0.0152,0.02,0.005021,0.01,0.02,190.60,100.0,...,93.739861,0.51,0.49,0.00,0.59,0.14,0.07,0.13,0.05,0.02
1,1,0.01,0.040,0.0392,0.04,0.006917,0.02,0.06,290.94,300.0,...,94.011927,0.49,0.43,0.08,0.51,0.07,0.21,0.10,0.06,0.05
2,2,0.01,0.005,0.0107,0.01,0.002564,0.01,0.02,1902.15,1510.0,...,151.864373,0.54,0.46,0.00,0.44,0.02,0.19,0.01,0.11,0.23
3,3,0.21,0.150,0.1310,0.10,0.046482,0.10,0.20,275.13,200.0,...,74.986447,0.50,0.49,0.01,0.43,0.18,0.09,0.11,0.18,0.01
4,4,0.02,0.020,0.0174,0.02,0.004408,0.01,0.02,143.15,122.5,...,126.849955,0.48,0.52,0.00,0.42,0.09,0.18,0.02,0.19,0.10


Save to csv

In [ ]:
features.to_csv(
    "/content/drive/MyDrive/Challengedata/eng_features.csv",
    index=False
)